In [23]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt
import numpy as np
import os
import urllib.request

In [2]:
# Device Configuration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device : {device}")

if device.type == "cuda":
    print(f"GPU Name     : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version : {torch.version.cuda}")
    print(f"GPU Count    : {torch.cuda.device_count()}")

Using Device : cuda
GPU Name     : NVIDIA GeForce GTX 1650
CUDA Version : 12.8
GPU Count    : 1


# Hyperparameters

In [3]:
BATCH_SIZE = 32
BLOCK_SIZE = 128
EMBED_SIZE = 256
NUM_LAYERS = 6
DROPOUT = 0.2
LEARNING_RATE = 3e-4
EPOCHS = 10

In [4]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

file_path = "data/input.txt"

if not os.path.exists(file_path): 
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, file_path)
    print("Download complete!")
else:
    print("Dataset already exists.")

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset Length: {len(text):,} characters")

Dataset already exists.
Dataset Length: 1,115,394 characters


In [5]:
char = sorted(list(set(text)))

vocav_size = len(char)
print(f"Vocabulary Size: {vocav_size}")
print(char)

Vocabulary Size: 65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [6]:
stoi = {ch: i for i, ch in enumerate(char)}
itos = {i: ch for i, ch in enumerate(char)}

print("Character -> Integer")
print(stoi)

print("Integer -> Character")
print(itos)

Character -> Integer
{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
Integer -> Character
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43

In [7]:
def encode(text):
    return [stoi[ch] for ch in text]

def decode(tokens):
    return "".join([itos[token] for token in tokens])

In [8]:
sample  = "mohit"
encoded = encode(sample)

print("Encoded: ",encoded)
print("Decoded: ", decode(encoded))

Encoded:  [51, 53, 46, 47, 58]
Decoded:  mohit


In [10]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data)
print()
print("shape: ", data.shape)
print("Data type: ", data.dtype)

tensor([18, 47, 56,  ..., 45,  8,  0])

shape:  torch.Size([1115394])
Data type:  torch.int64


In [11]:
x = data[:BLOCK_SIZE]
y = data[1:BLOCK_SIZE+1]

print("Input", x)
print("Target", y)

Input tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1])
Target tensor([47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44, 53,
        56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,  1,
        44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1, 57,
        54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,  6,
         1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47, 58,
        47

In [13]:
print("Input Text :")
print(decode(x.tolist()))

print()

print("Target Text :")
print(decode(y.tolist()))

Input Text :
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to 

Target Text :
irst Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to d


In [15]:
class TinyGPTDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.block_size]
        y = self.data[idx: 1:idx + self.block_size + 1]
        return x, y

In [17]:
dataset = TinyGPTDataset(data, BLOCK_SIZE)

print("Number of Samples:", len(dataset))

Number of Samples: 1115266


In [18]:
x, y = dataset[0]

print("Input IDs :", x)
print("Target IDs:", y)

Input IDs : tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1])
Target IDs: tensor([18])


In [20]:
print()

print("Input Text:")
print(decode(x.tolist()))

print()

print("Target Text:")
print(decode(y.tolist()))


Input Text:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to 

Target Text:
F


In [21]:
for i in range(len(dataset)):
    x, y = dataset[i]

In [24]:
train_loader = DataLoader(
    dataset,
    batch_size= BATCH_SIZE,
    shuffle=True
)

In [25]:
x_batch, y_batch = next(iter(train_loader))

print("Input Batch Shape :", x_batch.shape)
print("Target Batch Shape:", y_batch.shape)

Input Batch Shape : torch.Size([32, 128])
Target Batch Shape: torch.Size([32, 0])


In [27]:
print("Input:")
print(decode(x_batch[0].tolist()))

print()

print("Target:")
print(decode(y_batch[0].tolist()))

Input:
ifest appeal,
And put your trial in the villain's mouth
Which here you come to accuse.

LUCIO:
This is the rascal; this is he I 

Target:



# Embedding

In [29]:
class ScratchEmbedding(nn.Module):
    def __init__(self, vocav_size, embed_dim):
        super().__init__()

        self.weight = nn.Parameter(
            torch.randn(vocav_size, embed_dim)
        )

    def forward(self, token_ids):
        return self.weight[token_ids]